In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2) Setup paths
from pathlib import Path
from zipfile import ZipFile
import os

# Source ZIPs in Drive
DRIVE = Path("/content/drive/MyDrive/deepfake_project")
ZIP_TRAIN = DRIVE / "zips/dataset_train.zip"
ZIP_TEST  = DRIVE / "zips/dataset_test.zip"

# Destination folders in Colab
DST_AFHQ_TRAIN   = Path("/content/stargan-v2/combined_balanced/train")
DST_AFHQ_TEST    = Path("/content/stargan-v2/combined_balanced/test")



# Create destinations
DST_AFHQ_TRAIN.mkdir(parents=True, exist_ok=True)
DST_AFHQ_TEST.mkdir(parents=True, exist_ok=True)

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def safe_extract_images(zip_path: Path, dest_dir: Path):
    """
    Extract only image files from the zip to dest_dir (preserves internal subpaths),
    with path traversal protection.
    """
    if not zip_path.exists():
        raise FileNotFoundError(f"ZIP not found: {zip_path}")

    extracted = 0
    with ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            # Skip directories
            if info.is_dir():
                continue
            # Only extract images
            suffix = Path(info.filename).suffix.lower()
            if suffix not in IMG_EXT:
                continue

            # Path traversal protection
            target_path = dest_dir / info.filename
            target_path_parent = target_path.parent.resolve()
            dest_root_resolved = dest_dir.resolve()
            if not str(target_path_parent).startswith(str(dest_root_resolved)):
                # Skip anything that tries to escape the destination
                continue

            # Make sure subdirs exist
            target_path.parent.mkdir(parents=True, exist_ok=True)

            # Extract this single file
            with zf.open(info, "r") as src_f, open(target_path, "wb") as dst_f:
                dst_f.write(src_f.read())
            extracted += 1
    return extracted

def count_images(root: Path) -> int:
    return sum(1 for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT)

# 3) Extract train + test images into /content/stargan-v2/combined_balanced
n_train = safe_extract_images(ZIP_TRAIN, DST_AFHQ_TRAIN)
n_test  = safe_extract_images(ZIP_TEST,  DST_AFHQ_TEST)


print(f"[DONE] Extracted TRAIN images -> {DST_AFHQ_TRAIN}: {n_train} files")
print(f"[DONE] Extracted TEST  images -> {DST_AFHQ_TEST}: {n_test} files")

Mounted at /content/drive
[DONE] Extracted TRAIN images -> /content/stargan-v2/combined_balanced/train: 23040 files
[DONE] Extracted TEST  images -> /content/stargan-v2/combined_balanced/test: 11520 files


In [ ]:
!pip install shap lime --quiet

import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt
import numpy as np
import os

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
EXPLAIN_DIR = "/content/stargan-v2/explainability"
os.makedirs(EXPLAIN_DIR, exist_ok=True)

In [ ]:
from sklearn.metrics import roc_auc_score

def explain_xgb_with_shap(
    model,
    X_train,
    X_test,
    y_test,
    feature_names,
    prefix,
    max_bg=1000,
    max_test=2000,
    random_state=42
):
    """
    model         : fitted XGBoost (or other tree) model
    X_train       : numpy array or DataFrame (train features)
    X_test        : numpy array or DataFrame (test features)
    y_test        : test labels (0/1)
    feature_names : list of feature names (len == X.shape[1])
    prefix        : short string for filenames, e.g. "rq1_forensic"
    """
    print(f"\n=== SHAP explainability for {prefix} ===")

    # Ensure numpy arrays
    if isinstance(X_train, pd.DataFrame):
        X_train_np = X_train.values
    else:
        X_train_np = X_train

    if isinstance(X_test, pd.DataFrame):
        X_test_np = X_test.values
    else:
        X_test_np = X_test

    n_bg   = min(max_bg, X_train_np.shape[0])
    n_test = min(max_test, X_test_np.shape[0])

    rng = np.random.RandomState(random_state)
    bg_idx   = rng.choice(X_train_np.shape[0], size=n_bg, replace=False)
    test_idx = rng.choice(X_test_np.shape[0], size=n_test, replace=False)

    X_bg   = X_train_np[bg_idx]
    X_tsub = X_test_np[test_idx]
    y_tsub = np.array(y_test)[test_idx]

    # TreeExplainer is optimized for XGBoost / tree ensembles
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_tsub)

    # 1) Global summary (beeswarm)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_values,
        X_tsub,
        feature_names=feature_names,
        show=False
    )
    out_path = os.path.join(EXPLAIN_DIR, f"{prefix}_shap_summary_beeswarm.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(f"  - Saved SHAP beeswarm to {out_path}")

    # 2) Global bar plot of mean |SHAP|
    plt.figure(figsize=(8, 6))
    shap.summary_plot(
        shap_values,
        X_tsub,
        feature_names=feature_names,
        plot_type="bar",
        show=False
    )
    out_path = os.path.join(EXPLAIN_DIR, f"{prefix}_shap_importance_bar.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(f"  - Saved SHAP bar plot to {out_path}")

    # 3) Save mean |SHAP| as CSV for tables in the dissertation
    abs_mean = np.mean(np.abs(shap_values), axis=0)
    shap_df = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": abs_mean,
    }).sort_values("mean_abs_shap", ascending=False)
    out_csv = os.path.join(EXPLAIN_DIR, f"{prefix}_shap_importance.csv")
    shap_df.to_csv(out_csv, index=False)
    print(f"  - Saved SHAP importance CSV to {out_csv}")

    # 4) Optional: local explanations for a few test points
    # pick one TP, one FP, one FN if possible
    y_pred_proba = model.predict_proba(X_test_np)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    idx_tp = np.where((y_pred == 1) & (y_test == 1))[0]
    idx_fp = np.where((y_pred == 1) & (y_test == 0))[0]
    idx_fn = np.where((y_pred == 0) & (y_test == 1))[0]

    def _local_plot(idx, tag):
        x = X_test_np[idx:idx+1]
        shap_val = explainer.shap_values(x)
        plt.figure(figsize=(8, 4))
        shap.force_plot(
            explainer.expected_value,
            shap_val,
            x,
            feature_names=feature_names,
            matplotlib=True,
            show=False
        )
        out_path = os.path.join(EXPLAIN_DIR, f"{prefix}_local_{tag}_idx{idx}.png")
        plt.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close()
        print(f"  - Saved local SHAP force plot ({tag}) to {out_path}")

    if len(idx_tp) > 0:
        _local_plot(idx_tp[0], "tp")
    if len(idx_fp) > 0:
        _local_plot(idx_fp[0], "fp")
    if len(idx_fn) > 0:
        _local_plot(idx_fn[0], "fn")

    return shap_df


In [ ]:
ZIP_BOVW = DRIVE / "output/bovw3/stargan-v2_data_bovw_kmeans_pca-20260115-201211.zip"
DST_BOVW   = Path("/content/stargan-v2/bovw")
DST_BOVW.mkdir(parents=True, exist_ok=True)

def safe_extract_files(zip_path: Path, dest_dir: Path):
    """
    Extract only image files from the zip to dest_dir (preserves internal subpaths),
    with path traversal protection.
    """
    if not zip_path.exists():
        raise FileNotFoundError(f"ZIP not found: {zip_path}")

    extracted = 0
    with ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            # Skip directories
            if info.is_dir():
                continue


            # Path traversal protection
            target_path = dest_dir / info.filename
            target_path_parent = target_path.parent.resolve()
            dest_root_resolved = dest_dir.resolve()
            if not str(target_path_parent).startswith(str(dest_root_resolved)):
                # Skip anything that tries to escape the destination
                continue

            # Make sure subdirs exist
            target_path.parent.mkdir(parents=True, exist_ok=True)

            # Extract this single file
            with zf.open(info, "r") as src_f, open(target_path, "wb") as dst_f:
                dst_f.write(src_f.read())
            extracted += 1
    return extracted

n_bovw = safe_extract_files(ZIP_BOVW, DST_BOVW)
print(f"[DONE] Extracted BOVW files -> {DST_BOVW}: {n_bovw} files")

[DONE] Extracted BOVW files -> /content/stargan-v2/bovw: 23094 files


In [ ]:
!pip install -q xgboost

import time
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_recall_fscore_support
)
from xgboost import XGBClassifier

In [ ]:
BOVW_DIR = Path("/content/stargan-v2/bovw/bovw")

train_df = pd.read_csv(BOVW_DIR / "image_features.csv")
test_df  = pd.read_csv(BOVW_DIR / "image_features_test.csv")

print(train_df.shape, test_df.shape)
train_df.head()

(23040, 434) (11520, 434)


,path,label,label_name,species,fake_technique,width,height,aspect_ratio,brightness,contrast,...,prob_k50_40,prob_k50_41,prob_k50_42,prob_k50_43,prob_k50_44,prob_k50_45,prob_k50_46,prob_k50_47,prob_k50_48,prob_k50_49
0,/content/stargan-v2/combined_balanced/train/re...,0,real,cat,none,512,512,1.0,99.711258,46.337177,...,0.022608,0.024608,0.015885,0.036249,0.016437,0.019920,0.025904,0.021604,0.014518,0.021669
1,/content/stargan-v2/combined_balanced/train/re...,0,real,cat,none,512,512,1.0,117.625412,56.125246,...,0.021676,0.018423,0.015248,0.022378,0.017733,0.022555,0.026518,0.016763,0.015237,0.021479
2,/content/stargan-v2/combined_balanced/train/re...,0,real,cat,none,512,512,1.0,144.163239,73.633186,...,0.020257,0.016621,0.025911,0.016035,0.024569,0.019079,0.014860,0.019715,0.028191,0.014203
3,/content/stargan-v2/combined_balanced/train/re...,0,real,cat,none,512,512,1.0,90.459499,57.823762,...,0.022757,0.022062,0.020057,0.023534,0.022749,0.020329,0.017167,0.022550,0.019411,0.015656
4,/content/stargan-v2/combined_balanced/train/re...,0,real,cat,none,512,512,1.0,127.509350,43.617446,...,0.019280,0.019883,0.020185,0.024542,0.018004,0.021675,0.018992,0.027257,0.018821,0.015805


In [ ]:
y_train = train_df["label"].astype(int).values
y_test  = test_df["label"].astype(int).values


In [ ]:
from collections import defaultdict

def run_xgb_cv_and_test(X_train, y_train, X_test, y_test, model_name,
                        n_splits=5, random_state=42):
    """
    5-fold CV on TRAIN with XGBoost, then fit on full TRAIN and evaluate on TEST.
    Returns:
      final_model, oof_proba, test_proba, cv_summary, test_metrics
    """
    X_train = np.asarray(X_train, dtype=np.float32)
    X_test  = np.asarray(X_test,  dtype=np.float32)
    y_train = np.asarray(y_train, dtype=int)
    y_test  = np.asarray(y_test,  dtype=int)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_proba = np.zeros(len(y_train), dtype=float)
    fold_stats = []

    print(f"\n=== {model_name}: 5-fold CV ===")
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        model = XGBClassifier(
            n_estimators=1000,
            max_depth=10,
            learning_rate=0.001,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=random_state
        )

        t0 = time.time()
        model.fit(X_tr, y_tr)
        train_time = time.time() - t0

        val_proba = model.predict_proba(X_val)[:, 1]
        oof_proba[val_idx] = val_proba

        auc = roc_auc_score(y_val, val_proba)
        val_pred = (val_proba >= 0.5).astype(int)
        acc = accuracy_score(y_val, val_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_val, val_pred, average="binary", zero_division=0
        )

        fold_stats.append({
            "fold": fold,
            "auc": auc,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "train_time_s": train_time
        })

        print(f"  Fold {fold}: AUC={auc:.3f}, F1={f1:.3f}, Acc={acc:.3f}, "
              f"Prec={prec:.3f}, Rec={rec:.3f}, time={train_time:.2f}s")

    # CV summary
    cv_summary = {}
    for key in ["auc", "accuracy", "precision", "recall", "f1", "train_time_s"]:
        vals = [fs[key] for fs in fold_stats]
        cv_summary[f"{key}_mean"] = float(np.mean(vals))
        cv_summary[f"{key}_std"]  = float(np.std(vals))

    print("\nCV summary:")
    for k, v in cv_summary.items():
        print(f"  {k}: {v:.4f}")

    # Fit final model on full TRAIN and evaluate on TEST
    final_model = XGBClassifier(
        n_estimators=1000,
        max_depth=10,
        learning_rate=0.001,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=random_state
    )

    t0 = time.time()
    final_model.fit(X_train, y_train)
    full_train_time = time.time() - t0

    test_proba = final_model.predict_proba(X_test)[:, 1]
    test_pred  = (test_proba >= 0.5).astype(int)

    test_auc = roc_auc_score(y_test, test_proba)
    test_acc = accuracy_score(y_test, test_pred)
    test_prec, test_rec, test_f1, _ = precision_recall_fscore_support(
        y_test, test_pred, average="binary", zero_division=0
    )

    test_metrics = {
        "auc": float(test_auc),
        "accuracy": float(test_acc),
        "precision": float(test_prec),
        "recall": float(test_rec),
        "f1": float(test_f1),
        "full_train_time_s": float(full_train_time),
        "n_features": int(X_train.shape[1])
    }

    print(f"\n=== {model_name}: TEST metrics ===")
    print(f"  AUC={test_auc:.3f}, F1={test_f1:.3f}, Acc={test_acc:.3f}, "
          f"Prec={test_prec:.3f}, Rec={test_rec:.3f}")
    print(f"  Features={X_train.shape[1]}, Train_time_full={full_train_time:.2f}s")

    return final_model, oof_proba, test_proba, cv_summary, test_metrics


In [ ]:
forensic_cols = [
    "brightness", "contrast", "sharpness_l1_mean",
    "edge_density", "lap_var",
    "file_size"
]

X_train_forensic = train_df[forensic_cols].values
X_test_forensic  = test_df[forensic_cols].values

forensic_model, forensic_oof, forensic_test_proba, forensic_cv, forensic_test = \
    run_xgb_cv_and_test(X_train_forensic, y_train,
                        X_test_forensic, y_test,
                        model_name="RQ1_Forensic_XGB")

print("\nRQ1 forensic TEST metrics:", forensic_test)



=== RQ1_Forensic_XGB: 5-fold CV ===
  Fold 1: AUC=0.933, F1=0.847, Acc=0.841, Prec=0.814, Rec=0.884, time=25.90s
  Fold 2: AUC=0.924, F1=0.829, Acc=0.822, Prec=0.796, Rec=0.865, time=2.53s
  Fold 3: AUC=0.938, F1=0.850, Acc=0.845, Prec=0.826, Rec=0.875, time=2.08s
  Fold 4: AUC=0.935, F1=0.850, Acc=0.844, Prec=0.819, Rec=0.883, time=4.83s
  Fold 5: AUC=0.933, F1=0.847, Acc=0.842, Prec=0.822, Rec=0.875, time=2.05s

CV summary:
  auc_mean: 0.9324
  auc_std: 0.0047
  accuracy_mean: 0.8388
  accuracy_std: 0.0087
  precision_mean: 0.8152
  precision_std: 0.0105
  recall_mean: 0.8764
  recall_std: 0.0066
  f1_mean: 0.8447
  f1_std: 0.0078
  train_time_s_mean: 7.4783
  train_time_s_std: 9.2697

=== RQ1_Forensic_XGB: TEST metrics ===
  AUC=0.932, F1=0.844, Acc=0.838, Prec=0.812, Rec=0.880
  Features=6, Train_time_full=2.25s

RQ1 forensic TEST metrics: {'auc': 0.9323753978587964, 'accuracy': 0.8377604166666667, 'precision': 0.8116290245074483, 'recall': 0.8796875, 'f1': 0.844288927768058, 'ful

In [ ]:
# Example: RQ1 forensic model
# Assume:
#   xgb_forensic_full : fitted XGB model
#   X_forensic_train, X_forensic_test
#   y_train, y_test
#   forensic_cols: list of forensic feature column names

shap_rq1 = explain_xgb_with_shap(
    model=forensic_model,
    X_train=X_train_forensic,
    X_test=X_test_forensic,
    y_test=y_test,
    feature_names=forensic_cols,
    prefix="rq1_forensic_xgb"
)


=== SHAP explainability for rq1_forensic_xgb ===
  - Saved SHAP beeswarm to /content/stargan-v2/explainability/rq1_forensic_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/stargan-v2/explainability/rq1_forensic_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/stargan-v2/explainability/rq1_forensic_xgb_shap_importance.csv
  - Saved local SHAP force plot (tp) to /content/stargan-v2/explainability/rq1_forensic_xgb_local_tp_idx5760.png
  - Saved local SHAP force plot (fp) to /content/stargan-v2/explainability/rq1_forensic_xgb_local_fp_idx7.png
  - Saved local SHAP force plot (fn) to /content/stargan-v2/explainability/rq1_forensic_xgb_local_fn_idx6729.png


<Figure size 800x400 with 0 Axes>

<Figure size 800x400 with 0 Axes>

<Figure size 800x400 with 0 Axes>

In [ ]:
# grab all BoVW columns
bovw_cols = [c for c in train_df.columns if c.startswith("bovw_")]
print("BoVW dims:", len(bovw_cols))

# soft cluster probabilities (if present)
prob_k2_cols  = [c for c in train_df.columns if c.startswith("prob_k2_")]
prob_k10_cols = [c for c in train_df.columns if c.startswith("prob_k10_")]
prob_k50_cols = [c for c in train_df.columns if c.startswith("prob_k50_")]

print("prob_k2 dims:", len(prob_k2_cols),
      "prob_k10 dims:", len(prob_k10_cols),
      "prob_k50 dims:", len(prob_k50_cols))

bovw_k_feats = bovw_cols + prob_k2_cols + prob_k10_cols + prob_k50_cols

X_train_bovw_k = train_df[bovw_k_feats].values
X_test_bovw_k  = test_df[bovw_k_feats].values

bovw_k_model, bovw_k_oof, bovw_k_test_proba, bovw_k_cv, bovw_k_test = \
    run_xgb_cv_and_test(X_train_bovw_k, y_train,
                        X_test_bovw_k, y_test,
                        model_name="RQ2_BoVW_plus_Kmeans_XGB")

print("\nRQ2 BoVW+K TEST metrics:", bovw_k_test)


BoVW dims: 257
prob_k2 dims: 2 prob_k10 dims: 10 prob_k50 dims: 50

=== RQ2_BoVW_plus_Kmeans_XGB: 5-fold CV ===
  Fold 1: AUC=0.592, F1=0.535, Acc=0.566, Prec=0.577, Rec=0.498, time=170.42s
  Fold 2: AUC=0.586, F1=0.542, Acc=0.565, Prec=0.572, Rec=0.516, time=172.78s
  Fold 3: AUC=0.579, F1=0.530, Acc=0.554, Prec=0.560, Rec=0.503, time=174.50s
  Fold 4: AUC=0.583, F1=0.544, Acc=0.558, Prec=0.561, Rec=0.528, time=173.88s
  Fold 5: AUC=0.599, F1=0.551, Acc=0.569, Prec=0.575, Rec=0.529, time=172.47s

CV summary:
  auc_mean: 0.5879
  auc_std: 0.0071
  accuracy_mean: 0.5625
  accuracy_std: 0.0056
  precision_mean: 0.5692
  precision_std: 0.0069
  recall_mean: 0.5148
  recall_std: 0.0124
  f1_mean: 0.5405
  f1_std: 0.0073
  train_time_s_mean: 172.8101
  train_time_s_std: 1.4010

=== RQ2_BoVW_plus_Kmeans_XGB: TEST metrics ===
  AUC=0.646, F1=0.559, Acc=0.603, Prec=0.629, Rec=0.503
  Features=319, Train_time_full=186.39s

RQ2 BoVW+K TEST metrics: {'auc': 0.6456857186776621, 'accuracy': 0.60303

In [ ]:
# RQ2: BoVW+KMeans model
shap_rq2 = explain_xgb_with_shap(
    model=bovw_k_model,
    X_train=X_train_bovw_k,
    X_test=X_test_bovw_k,
    y_test=y_test,
    feature_names=bovw_k_feats,  # e.g. ["bovw_0", ..., "prob_k2_0", "prob_k10_0", ...]
    prefix="rq2_bovw_k_xgb"
)


=== SHAP explainability for rq2_bovw_k_xgb ===
  - Saved SHAP beeswarm to /content/stargan-v2/explainability/rq2_bovw_k_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/stargan-v2/explainability/rq2_bovw_k_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/stargan-v2/explainability/rq2_bovw_k_xgb_shap_importance.csv
  - Saved local SHAP force plot (tp) to /content/stargan-v2/explainability/rq2_bovw_k_xgb_local_tp_idx5760.png
  - Saved local SHAP force plot (fp) to /content/stargan-v2/explainability/rq2_bovw_k_xgb_local_fp_idx6.png
  - Saved local SHAP force plot (fn) to /content/stargan-v2/explainability/rq2_bovw_k_xgb_local_fn_idx5837.png


In [ ]:
sil_df = pd.read_csv(BOVW_DIR / "silhouette_k_grid.csv")
print("Silhouette grid:\n", sil_df)
best_k = sil_df.iloc[sil_df["silhouette"].idxmax()]["k"]
print("Best k by silhouette:", best_k)


Silhouette grid:
     k  silhouette
0   2    0.127419
1   3    0.107699
2   5    0.083397
3  10    0.059630
4  20    0.047294
5  50    0.025098
Best k by silhouette: 2.0


In [ ]:
# Raw BoVW only
X_train_bovw = train_df[bovw_cols].values
X_test_bovw  = test_df[bovw_cols].values

bovw_model, bovw_oof, bovw_test_proba, bovw_cv, bovw_test = \
    run_xgb_cv_and_test(X_train_bovw, y_train,
                        X_test_bovw, y_test,
                        model_name="RQ3_BoVW_raw_XGB")

# PCA features only
pca_cols = [c for c in train_df.columns if c.startswith("pca_")]
X_train_pca = train_df[pca_cols].values
X_test_pca  = test_df[pca_cols].values

pca_model, pca_oof, pca_test_proba, pca_cv, pca_test = \
    run_xgb_cv_and_test(X_train_pca, y_train,
                        X_test_pca, y_test,
                        model_name="RQ3_BoVW_PCA_XGB")

print("\nRQ3 BoVW raw TEST metrics:", bovw_test)
print("RQ3 BoVW PCA TEST metrics:", pca_test)



=== RQ3_BoVW_raw_XGB: 5-fold CV ===
  Fold 1: AUC=0.583, F1=0.533, Acc=0.564, Prec=0.574, Rec=0.497, time=146.54s
  Fold 2: AUC=0.579, F1=0.529, Acc=0.560, Prec=0.569, Rec=0.494, time=150.17s
  Fold 3: AUC=0.564, F1=0.511, Acc=0.535, Prec=0.539, Rec=0.486, time=154.88s
  Fold 4: AUC=0.579, F1=0.533, Acc=0.556, Prec=0.562, Rec=0.507, time=153.51s
  Fold 5: AUC=0.587, F1=0.539, Acc=0.564, Prec=0.572, Rec=0.510, time=150.45s

CV summary:
  auc_mean: 0.5785
  auc_std: 0.0077
  accuracy_mean: 0.5559
  accuracy_std: 0.0107
  precision_mean: 0.5633
  precision_std: 0.0127
  recall_mean: 0.4988
  recall_std: 0.0088
  f1_mean: 0.5290
  f1_std: 0.0096
  train_time_s_mean: 151.1114
  train_time_s_std: 2.9067

=== RQ3_BoVW_raw_XGB: TEST metrics ===
  AUC=0.644, F1=0.558, Acc=0.604, Prec=0.631, Rec=0.500
  Features=257, Train_time_full=163.88s

=== RQ3_BoVW_PCA_XGB: 5-fold CV ===
  Fold 1: AUC=0.602, F1=0.531, Acc=0.575, Prec=0.592, Rec=0.481, time=58.57s
  Fold 2: AUC=0.587, F1=0.525, Acc=0.565, 

In [ ]:
# RQ3: BoVW-raw
shap_rq3_raw = explain_xgb_with_shap(
    model=bovw_model,
    X_train=X_train_bovw,
    X_test=X_test_bovw,
    y_test=y_test,
    feature_names=bovw_cols,

    prefix="rq3_bovw_raw_xgb"
)


=== SHAP explainability for rq3_bovw_raw_xgb ===
  - Saved SHAP beeswarm to /content/stargan-v2/explainability/rq3_bovw_raw_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/stargan-v2/explainability/rq3_bovw_raw_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/stargan-v2/explainability/rq3_bovw_raw_xgb_shap_importance.csv
  - Saved local SHAP force plot (tp) to /content/stargan-v2/explainability/rq3_bovw_raw_xgb_local_tp_idx5760.png
  - Saved local SHAP force plot (fp) to /content/stargan-v2/explainability/rq3_bovw_raw_xgb_local_fp_idx13.png
  - Saved local SHAP force plot (fn) to /content/stargan-v2/explainability/rq3_bovw_raw_xgb_local_fn_idx5767.png


In [ ]:
# RQ3: BoVW+PCA
shap_rq3_pca = explain_xgb_with_shap(
    model=pca_model,
    X_train=X_train_pca,
    X_test=X_test_pca,
    y_test=y_test,
    feature_names=pca_cols,  # e.g. ["pca_0", ..., "pca_63"]
    prefix="rq3_bovw_pca_xgb"
)


=== SHAP explainability for rq3_bovw_pca_xgb ===
  - Saved SHAP beeswarm to /content/stargan-v2/explainability/rq3_bovw_pca_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/stargan-v2/explainability/rq3_bovw_pca_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/stargan-v2/explainability/rq3_bovw_pca_xgb_shap_importance.csv
  - Saved local SHAP force plot (tp) to /content/stargan-v2/explainability/rq3_bovw_pca_xgb_local_tp_idx5760.png
  - Saved local SHAP force plot (fp) to /content/stargan-v2/explainability/rq3_bovw_pca_xgb_local_fp_idx0.png
  - Saved local SHAP force plot (fn) to /content/stargan-v2/explainability/rq3_bovw_pca_xgb_local_fn_idx5792.png


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import glob, os

IMG_SIZE = (256, 256)
BATCH_SIZE = 64

def list_images_with_labels(root_dir):
    paths, labels = [], []
    for label_name, label_val in [("real", 0), ("fake", 1)]:
        for p in glob.glob(os.path.join(root_dir, label_name, "**", "*.*"), recursive=True):
            ext = os.path.splitext(p)[1].lower()
            if ext in [".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"]:
                paths.append(p)
                labels.append(label_val)
    return np.array(paths), np.array(labels)

train_img_paths, train_img_labels = list_images_with_labels(str(DST_AFHQ_TRAIN))
test_img_paths,  test_img_labels  = list_images_with_labels(str(DST_AFHQ_TEST))

print("Train images:", len(train_img_paths), "Test images:", len(test_img_paths))


Train images: 23040 Test images: 11520


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE   = (256, 256)
INPUT_SHAPE = IMG_SIZE + (3,)

def build_small_cnn(input_shape=INPUT_SHAPE):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # --- Optional augmentation (can comment out if you want) ---
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        # -----------------------------------------------------------

        layers.Rescaling(1./255),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.GlobalAveragePooling2D(),

        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )
    return model



In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 64

def decode_and_resize(path, label):
    # path is a tf.string
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    # model will rescale to [0,1], so keep as uint8/float32 here
    img = tf.cast(img, tf.float32)
    label = tf.cast(label, tf.float32)
    return img, label

def make_dataset(paths, labels, shuffle=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
import time

def run_cnn_cv_and_test(train_paths, train_labels,
                        test_paths, test_labels,
                        n_splits=5, epochs=16, batch_size=64,
                        random_state=42):

    train_paths = np.array(train_paths)
    train_labels = np.array(train_labels).astype(int)
    test_paths = np.array(test_paths)
    test_labels = np.array(test_labels).astype(int)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_proba = np.zeros(len(train_labels), dtype=float)
    fold_stats = []

    print("\n=== CNN: 5-fold CV ===")
    fold_idx = 0

    for tr_idx, val_idx in skf.split(train_paths, train_labels):
        fold_idx += 1
        print(f"\n--- Fold {fold_idx}/{n_splits} ---")

        tr_paths, val_paths = train_paths[tr_idx], train_paths[val_idx]
        tr_labels, val_labels = train_labels[tr_idx], train_labels[val_idx]

        train_ds = make_dataset(tr_paths, tr_labels, shuffle=True,  batch_size=batch_size)
        val_ds   = make_dataset(val_paths, val_labels, shuffle=False, batch_size=batch_size)

        model = build_small_cnn()

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=2, restore_best_weights=True
            )
        ]

        t0 = time.time()
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=1,
            callbacks=callbacks
        )
        train_time = time.time() - t0

        # Get probabilities on validation set
        val_proba = model.predict(val_ds).ravel()  # sigmoid outputs ∈ [0,1]
        oof_proba[val_idx] = val_proba

        val_pred = (val_proba >= 0.5).astype(int)

        fold_auc = roc_auc_score(val_labels, val_proba)
        fold_acc = accuracy_score(val_labels, val_pred)
        fold_prec, fold_rec, fold_f1, _ = precision_recall_fscore_support(
            val_labels, val_pred, average="binary", zero_division=0
        )

        print(f"Fold {fold_idx} AUC={fold_auc:.3f}, F1={fold_f1:.3f}, "
              f"Acc={fold_acc:.3f}, Prec={fold_prec:.3f}, Rec={fold_rec:.3f}, "
              f"train_time={train_time:.2f}s")

        fold_stats.append({
            "fold": fold_idx,
            "auc": fold_auc,
            "accuracy": fold_acc,
            "precision": fold_prec,
            "recall": fold_rec,
            "f1": fold_f1,
            "train_time_s": train_time
        })

    # CV summary
    cv_summary = {}
    for key in ["auc", "accuracy", "precision", "recall", "f1", "train_time_s"]:
        vals = [fs[key] for fs in fold_stats]
        cv_summary[f"{key}_mean"] = float(np.mean(vals))
        cv_summary[f"{key}_std"]  = float(np.std(vals))

    print("\n=== CNN CV summary ===")
    for k, v in cv_summary.items():
        print(f"  {k}: {v:.4f}")

    # === Train final model on full TRAIN and evaluate on TEST ===
    full_train_ds = make_dataset(train_paths, train_labels, shuffle=True, batch_size=batch_size)
    test_ds       = make_dataset(test_paths,  test_labels,  shuffle=False, batch_size=batch_size)

    final_model = build_small_cnn()

    t0 = time.time()
    final_model.fit(full_train_ds, epochs=epochs, verbose=1)
    full_train_time = time.time() - t0

    # Inference time
    t0 = time.time()
    test_proba = final_model.predict(test_ds).ravel()
    inference_time = time.time() - t0
    time_per_image = inference_time / len(test_labels)

    test_pred = (test_proba >= 0.5).astype(int)

    test_auc = roc_auc_score(test_labels, test_proba)
    test_acc = accuracy_score(test_labels, test_pred)
    test_prec, test_rec, test_f1, _ = precision_recall_fscore_support(
        test_labels, test_pred, average="binary", zero_division=0
    )

    test_metrics = {
        "auc": float(test_auc),
        "accuracy": float(test_acc),
        "precision": float(test_prec),
        "recall": float(test_rec),
        "f1": float(test_f1),
        "full_train_time_s": float(full_train_time),
        "inference_time_s_total": float(inference_time),
        "inference_time_s_per_image": float(time_per_image),
        "n_params": int(final_model.count_params())
    }

    print("\n=== CNN TEST metrics ===")
    print(f"  AUC={test_auc:.3f}, F1={test_f1:.3f}, Acc={test_acc:.3f}, "
          f"Prec={test_prec:.3f}, Rec={test_rec:.3f}")
    print(f"  Train_time_full={full_train_time:.2f}s, "
          f"Total_infer_time={inference_time:.2f}s, "
          f"Per_image={time_per_image*1000:.2f} ms")
    print(f"  Model parameters: {final_model.count_params()}")

    return final_model, oof_proba, test_proba, cv_summary, test_metrics


In [ ]:
cnn_model, cnn_oof_proba, cnn_test_proba, cnn_cv, cnn_test = \
    run_cnn_cv_and_test(train_img_paths, train_img_labels,
                        test_img_paths,  test_img_labels,
                        n_splits=5, epochs=16, batch_size=64)

print("\nCNN CV summary:", cnn_cv)
print("CNN TEST metrics:", cnn_test)



=== CNN: 5-fold CV ===

--- Fold 1/5 ---
Epoch 1/16
288/288 ━━━━━━━━━━━━━━━━━━━━ 36s 48ms/step - accuracy: 0.5008 - auc: 0.5019 - loss: 0.6934 - val_accuracy: 0.5197 - val_auc: 0.5024 - val_loss: 0.6930
Epoch 2/16
288/288 ━━━━━━━━━━━━━━━━━━━━ 28s 44ms/step - accuracy: 0.5061 - auc: 0.4977 - loss: 0.6932 - val_accuracy: 0.5043 - val_auc: 0.5050 - val_loss: 0.6929
Epoch 3/16
288/288 ━━━━━━━━━━━━━━━━━━━━ 26s 44ms/step - accuracy: 0.4965 - auc: 0.4991 - loss: 0.6933 - val_accuracy: 0.5011 - val_auc: 0.5007 - val_loss: 0.6931
Epoch 4/16
288/288 ━━━━━━━━━━━━━━━━━━━━ 26s 44ms/step - accuracy: 0.5054 - auc: 0.4981 - loss: 0.6931 - val_accuracy: 0.5000 - val_auc: 0.5146 - val_loss: 0.6959
72/72 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step
Fold 1 AUC=0.515, F1=0.634, Acc=0.504, Prec=0.503, Rec=0.857, train_time=115.53s

--- Fold 2/5 ---
Epoch 1/16
288/288 ━━━━━━━━━━━━━━━━━━━━ 29s 45ms/step - accuracy: 0.4869 - auc: 0.4879 - loss: 0.6939 - val_accuracy: 0.5000 - val_auc: 0.4993 - val_loss: 0.6932
Epoch 2/1

In [ ]:
import numpy as np
import tensorflow as tf

#train_img_paths, train_img_labels = list_images_with_labels(str(DST_AFHQ_TRAIN))
#test_img_paths,  test_img_labels  = list_images_with_labels(str(DST_AFHQ_TEST))

# Load all test images into memory for LIME (ok for ~10k images at 128x128)
def load_image_raw(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img.set_shape((*IMG_SIZE, 3))
    return img.numpy()  # float32, 0-255

X_test_imgs = np.stack([load_image_raw(p) for p in test_img_paths])
y_test = np.array(test_img_labels, dtype=int)
print("X_test_imgs:", X_test_imgs.shape, "y_test:", y_test.shape)


X_test_imgs: (11520, 256, 256, 3) y_test: (11520,)


In [ ]:
def cnn_classifier_fn(images: np.ndarray):
    """
    images: (N, H, W, 3) in 0-255 (uint8 or float).
    Returns probs for shape (N, 2): [p(real), p(fake)].
    """
    imgs = tf.convert_to_tensor(images, dtype=tf.float32)
    preds = cnn_model.predict(imgs, verbose=0).reshape(-1)  # sigmoid -> p(fake)
    preds_fake = preds
    preds_real = 1.0 - preds_fake
    return np.vstack([preds_real, preds_fake]).T


In [ ]:
from sklearn.metrics import roc_auc_score

# y_cnn_test_proba, y_cnn_test_pred already computed earlier
print("CNN Test AUC:", roc_auc_score(y_test, cnn_test_proba))

cnn_test_pred_soft = (cnn_test_proba >= 0.1).astype(int)

tp_idx = np.where((y_test == 1) & (cnn_test_proba == 1))[0]
tn_idx = np.where((y_test == 0) & (cnn_test_proba == 1))[0]
fp_idx = np.where((y_test == 0) & (cnn_test_proba == 0))[0]
fn_idx = np.where((y_test == 1) & (cnn_test_proba == 0))[0]


CNN Test AUC: 0.7718816008391204


In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt
from pathlib import Path

lime_img_explainer = lime_image.LimeImageExplainer()

EXPLAIN_IMG_DIR = Path("/content/stargan-v2/explainability/lime_cnn_images")
EXPLAIN_IMG_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)

def pick_some(idxs, k=2):
    idxs = np.array(idxs)
    if len(idxs) == 0:
        return []
    if len(idxs) <= k:
        return idxs.tolist()
    return rng.choice(idxs, size=k, replace=False).tolist()

def explain_cnn_image(idx, label_name_map=None):
    # image + meta
    img = X_test_imgs[idx].astype(np.uint8)
    true_label = y_test[idx]
    pred_prob = float(cnn_test_proba[idx])
    pred_label = int(pred_prob >= 0.5)
    path = test_img_paths[idx]
    fake_tech = test_df.iloc[idx]["fake_technique"] if "fake_technique" in test_df.columns else "unknown"
    label_name = test_df.iloc[idx]["label_name"] if "label_name" in test_df.columns else str(true_label)

    print("="*80)
    print(f"CNN LIME for test index {idx}")
    print(f"path          : {path}")
    print(f"true label    : {true_label} ({label_name})")
    print(f"pred label    : {pred_label} ({'fake' if pred_label==1 else 'real'})")
    print(f"pred prob(fake): {pred_prob:.3f}")
    if true_label == 1:
        print(f"fake_technique: {fake_tech}")
    print("-"*80)

    explanation = lime_img_explainer.explain_instance(
        image=img,
        classifier_fn=cnn_classifier_fn,
        top_labels=2,
        hide_color=0,
        num_samples=1000
    )

    # For the "fake" class (1)
    temp, mask = explanation.get_image_and_mask(
        label=1,
        positive_only=True,
        num_features=10,
        hide_rest=False
    )

    # Plot & save overlay
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(img)
    ax[0].set_title("Original")
    ax[0].axis("off")

    ax[1].imshow(mark_boundaries(temp / 255.0, mask))
    ax[1].set_title("LIME – regions supporting 'fake'")
    ax[1].axis("off")

    fname = EXPLAIN_IMG_DIR / f"lime_cnn_idx{idx}_true{true_label}_pred{pred_label}.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=200)
    plt.close()
    print(f"Saved LIME image explanation to {fname}")
    print()


In [ ]:
#example_groups = {
#    "TP_fake":        pick_some(tp_idx,  k=2),
#    "TN_real":        pick_some(tn_idx,  k=2),
#    "FP_real_as_fake": pick_some(fp_idx, k=2),
#    "FN_fake_as_real": pick_some(fn_idx, k=2),
#}

#for group_name, idxs in example_groups.items():
#    print(f"\n### {group_name} examples")
#    if not idxs:
#        print("  (none found)")
#        continue
#    for idx in idxs:
#        explain_cnn_image(idx)


In [ ]:
import numpy as np

# Flatten proba if needed
scores = cnn_test_proba.ravel()

# Indices of real / fake
idx_real = np.where(y_test == 0)[0]
idx_fake = np.where(y_test == 1)[0]

def top_k(idxs, score_vec, k, reverse=False):
    """Return up to k indices from idxs ordered by score."""
    idxs = np.array(idxs)
    if len(idxs) == 0:
        return []
    order = np.argsort(score_vec[idxs])
    if reverse:  # highest scores first
        order = order[::-1]
    sel = idxs[order[:k]]
    return sel.tolist()

# "Most confident" groups, even if model is bad
TP_like = top_k(idx_fake, scores, k=5, reverse=True)   # true fake, highest P(fake)
FN_like = top_k(idx_fake, scores, k=5, reverse=False)  # true fake, lowest P(fake)
TN_like = top_k(idx_real, scores, k=5, reverse=False)  # true real, lowest P(fake)
FP_like = top_k(idx_real, scores, k=5, reverse=True)   # true real, highest P(fake)

example_groups = {
    "TP_like_fake":        TP_like,
    "TN_like_real":        TN_like,
    "FP_real_as_fake":     FP_like,
    "FN_fake_as_real":     FN_like,
}

for group_name, idxs in example_groups.items():
    print(f"\n### {group_name} examples")
    if not idxs:
        print("  (none found)")
        continue
    for idx in idxs:
        explain_cnn_image(idx)



### TP_like_fake examples
CNN LIME for test index 6888
path          : /content/stargan-v2/combined_balanced/test/fake/swap_like/swap_like__927e2f2d__flickr_wild_000986_swap_3db43542__flickr_wild_002192.png
true label    : 1 (fake)
pred label    : 1 (fake)
pred prob(fake): 1.000
fake_technique: copy_move
--------------------------------------------------------------------------------


  0%|          | 0/1000 [00:00<?, ?it/s]

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_idx6888_true1_pred1.png

CNN LIME for test index 6585
path          : /content/stargan-v2/combined_balanced/test/fake/splicing/splicing__4d9dd0de__pixabay_cat_004697_splice_e80a860d__flickr_wild_002289.png
true label    : 1 (fake)
pred label    : 1 (fake)
pred prob(fake): 1.000
fake_technique: stargan_tiles
--------------------------------------------------------------------------------


  0%|          | 0/1000 [00:00<?, ?it/s]

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_idx6585_true1_pred1.png

CNN LIME for test index 7181
path          : /content/stargan-v2/combined_balanced/test/fake/swap_like/swap_like__360a3791__pixabay_dog_003074_swap_f2e7015d__pixabay_dog_000659.png
true label    : 1 (fake)
pred label    : 1 (fake)
pred prob(fake): 1.000
fake_technique: copy_move
--------------------------------------------------------------------------------


...

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_idx5974_true1_pred0.png

CNN LIME for test index 10467
path          : /content/stargan-v2/combined_balanced/test/fake/copy_move/copy_move__4ac5f1bc__pixabay_wild_000834_copymove.png
true label    : 1 (fake)
pred label    : 0 (real)
pred prob(fake): 0.109
fake_technique: splicing
--------------------------------------------------------------------------------


  0%|          | 0/1000 [00:00<?, ?it/s]

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_idx10467_true1_pred0.png

CNN LIME for test index 5958
path          : /content/stargan-v2/combined_balanced/test/fake/splicing/splicing__cd163de0__pixabay_dog_001433_splice_0ab6f11b__pixabay_dog_002695.png
true label    : 1 (fake)
pred label    : 0 (real)
pred prob(fake): 0.114
fake_technique: stargan_tiles
--------------------------------------------------------------------------------


  0%|          | 0/1000 [00:00<?, ?it/s]

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_idx5958_true1_pred0.png



In [ ]:
# Ensemble

ensemble_X_train = np.vstack([
    forensic_oof,
    bovw_k_oof,
    pca_oof,
    cnn_oof_proba  # when you have it
]).T

ensemble_X_test = np.vstack([
    forensic_test_proba,
    bovw_k_test_proba,
    pca_test_proba,
    cnn_test_proba  # when you have it
]).T

ens_model, ens_oof, ens_test_proba, ens_cv, ens_test = \
    run_xgb_cv_and_test(ensemble_X_train, y_train,
                        ensemble_X_test, y_test,
                        model_name="RQ5_Ensemble_XGB")

print("\nRQ5 ensemble TEST metrics:", ens_test)



=== RQ5_Ensemble_XGB: 5-fold CV ===
  Fold 1: AUC=0.928, F1=0.846, Acc=0.829, Prec=0.770, Rec=0.940, time=4.10s
  Fold 2: AUC=0.924, F1=0.831, Acc=0.824, Prec=0.799, Rec=0.866, time=2.01s
  Fold 3: AUC=0.933, F1=0.847, Acc=0.837, Prec=0.801, Rec=0.898, time=1.97s
  Fold 4: AUC=0.936, F1=0.850, Acc=0.845, Prec=0.822, Rec=0.881, time=2.04s
  Fold 5: AUC=0.933, F1=0.847, Acc=0.842, Prec=0.821, Rec=0.874, time=1.97s

CV summary:
  auc_mean: 0.9307
  auc_std: 0.0042
  accuracy_mean: 0.8355
  accuracy_std: 0.0076
  precision_mean: 0.8026
  precision_std: 0.0190
  recall_mean: 0.8916
  recall_std: 0.0262
  f1_mean: 0.8442
  f1_std: 0.0065
  train_time_s_mean: 2.4171
  train_time_s_std: 0.8432

=== RQ5_Ensemble_XGB: TEST metrics ===
  AUC=0.954, F1=0.879, Acc=0.872, Prec=0.838, Rec=0.923
  Features=4, Train_time_full=4.78s

RQ5 ensemble TEST metrics: {'auc': 0.9541965663580246, 'accuracy': 0.8723958333333334, 'precision': 0.8379017013232514, 'recall': 0.9234375, 'f1': 0.8785926660059464, 'ful

In [ ]:
# RQ5: Ensemble (forensic + BoVW + PCA + cluster probs, etc.)
# Example: after you build the proba columns

ensemble_X_train = pd.DataFrame({
    "forensic_proba": forensic_oof,
    "bovw_k_proba":   bovw_k_oof,
    "pca_proba":      pca_oof,
    "cnn_proba":      cnn_oof_proba,
})

ensemble_X_test = pd.DataFrame({
    "forensic_proba": forensic_test_proba,
    "bovw_k_proba":   bovw_k_test_proba,
    "pca_proba":      pca_test_proba,
    "cnn_proba":      cnn_test_proba,
})

shap_rq5_ens = explain_xgb_with_shap(
    model=ens_model,
    X_train=ensemble_X_train,
    X_test=ensemble_X_test,
    y_test=y_test,
    feature_names=list(ensemble_X_train.columns),
    # ["forensic_test_proba","bovw_k_test_proba", "pca_test_proba", "cnn_test_proba"],  # e.g. ["forensic_score","bovw_pca_score","prob_k2_1","prob_k10_4",...]
    prefix="rq5_ensemble_xgb"
)


=== SHAP explainability for rq5_ensemble_xgb ===
  - Saved SHAP beeswarm to /content/stargan-v2/explainability/rq5_ensemble_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/stargan-v2/explainability/rq5_ensemble_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/stargan-v2/explainability/rq5_ensemble_xgb_shap_importance.csv
  - Saved local SHAP force plot (tp) to /content/stargan-v2/explainability/rq5_ensemble_xgb_local_tp_idx5760.png
  - Saved local SHAP force plot (fp) to /content/stargan-v2/explainability/rq5_ensemble_xgb_local_fp_idx13.png
  - Saved local SHAP force plot (fn) to /content/stargan-v2/explainability/rq5_ensemble_xgb_local_fn_idx6737.png


In [ ]:
import numpy as np
import pandas as pd

# Where to save lift tables
LIFT_OUT_DIR = DRIVE / "output" / "lift_tables"
LIFT_OUT_DIR.mkdir(parents=True, exist_ok=True)

def compute_lift_table(
    y_true,
    y_score,
    base_df,
    model_name: str,
    n_bins: int = 10,
    out_dir: Path = LIFT_OUT_DIR,
):
    """
    Build a decile-based lift table on the TEST set.

    y_true  : array-like of 0/1 labels (1 = deepfake)
    y_score : array-like of predicted probabilities for class 1 (deepfake)
    base_df : DataFrame containing at least columns: 'label', 'label_name', 'fake_technique'
    model_name : string used in the output CSV file name
    n_bins  : number of bins (10 for deciles)
    out_dir : directory for CSV output

    Returns: lift_df (pandas DataFrame)
    """
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    if y_true.shape[0] != y_score.shape[0]:
        raise ValueError("y_true and y_score must have the same length")

    df = base_df.copy().reset_index(drop=True)
    if len(df) != len(y_true):
        raise ValueError("base_df length must match y_true/y_score length")

    df["y_true"] = y_true
    df["score"] = y_score

    # Sort by descending predicted probability (highest-risk first)
    df = df.sort_values("score", ascending=False).reset_index(drop=True)
    N = len(df)
    df["rank"] = np.arange(1, N + 1)

    # Assign deciles: 1 = top 10% highest score, 10 = lowest 10%
    df["decile"] = ((df["rank"] - 1) / (N / n_bins)).astype(int) + 1
    df.loc[df["decile"] > n_bins, "decile"] = n_bins

    # Overall target rate (deepfake rate) for lift calculations
    base_rate = df["y_true"].mean()
    total_pos = df["y_true"].sum()

    # Basic per-decile stats
    grouped = df.groupby("decile", as_index=False)
    lift_df = grouped["y_true"].agg(
        n="size",
        n_fake="sum",
    )
    lift_df["n_real"] = lift_df["n"] - lift_df["n_fake"]
    lift_df["pct_fake"] = lift_df["n_fake"] / lift_df["n"].replace(0, np.nan)
    lift_df["pct_real"] = lift_df["n_real"] / lift_df["n"].replace(0, np.nan)

    # Cumulative stats and lift
    lift_df["cum_n"] = lift_df["n"].cumsum()
    lift_df["cum_fake"] = lift_df["n_fake"].cumsum()
    lift_df["cum_fake_rate"] = lift_df["cum_fake"] / (total_pos if total_pos > 0 else np.nan)
    lift_df["pop_frac"] = lift_df["cum_n"] / float(N)

    # Per-decile lift vs overall deepfake rate
    lift_df["lift"] = lift_df["pct_fake"] / (base_rate if base_rate > 0 else np.nan)
    # Cumulative lift (how much better than random by this decile)
    lift_df["cum_lift"] = lift_df["cum_fake_rate"] / lift_df["pop_frac"].replace(0, np.nan)

    # Per-deepfake-method breakdown (counts and % among fakes in that decile)
    if "fake_technique" in df.columns:
        # Only look at fakes for technique stats
        fake_only = df[df["y_true"] == 1].copy()
        if not fake_only.empty:
            tech_tab = (
                fake_only
                .pivot_table(
                    index="decile",
                    columns="fake_technique",
                    values="y_true",
                    aggfunc="size",
                    fill_value=0,
                )
            )
            # Counts per technique
            tech_tab_counts = tech_tab.add_prefix("n_fake_")
            tech_tab_counts.reset_index(inplace=True)

            # Merge counts into lift_df
            lift_df = lift_df.merge(tech_tab_counts, on="decile", how="left")
            lift_df.fillna(0, inplace=True)

            # Percent-of-fakes per decile for each technique
            fake_cols = [c for c in lift_df.columns if c.startswith("n_fake_")]
            for c in fake_cols:
                pct_col = c.replace("n_fake_", "pct_fake_")
                lift_df[pct_col] = lift_df[c] / lift_df["n_fake"].replace(0, np.nan)

    # Save to CSV
    out_path = out_dir / f"lift_{model_name}_test.csv"
    lift_df.to_csv(out_path, index=False)
    print(f"[LIFT] Saved {model_name} lift table to: {out_path}")

    return lift_df


In [ ]:
# === Build lift tables on TEST for each model ===

# Sanity check that all proba arrays line up with test_df
print("Test length:", len(test_df))
print("  y_test:", len(y_test))
print("  forensic_test_proba:", len(forensic_test_proba))
print("  bovw_k_test_proba:", len(bovw_k_test_proba))
print("  bovw_test_proba:", len(bovw_test_proba))
print("  pca_test_proba:", len(pca_test_proba))
print("  cnn_test_proba:", len(cnn_test_proba))
print("  ens_test_proba:", len(ens_test_proba))

lift_rq1_forensic = compute_lift_table(
    y_true=y_test,
    y_score=forensic_test_proba,
    base_df=test_df,
    model_name="RQ1_Forensic_XGB",
)

lift_rq2_bovw_k = compute_lift_table(
    y_true=y_test,
    y_score=bovw_k_test_proba,
    base_df=test_df,
    model_name="RQ2_BoVW_plus_Kmeans_XGB",
)

lift_rq3_bovw_raw = compute_lift_table(
    y_true=y_test,
    y_score=bovw_test_proba,
    base_df=test_df,
    model_name="RQ3_BoVW_raw_XGB",
)

lift_rq3_bovw_pca = compute_lift_table(
    y_true=y_test,
    y_score=pca_test_proba,
    base_df=test_df,
    model_name="RQ3_BoVW_PCA_XGB",
)

lift_rq4_cnn = compute_lift_table(
    y_true=y_test,
    y_score=cnn_test_proba,
    base_df=test_df,
    model_name="RQ4_CNN",
)

lift_rq5_ensemble = compute_lift_table(
    y_true=y_test,
    y_score=ens_test_proba,
    base_df=test_df,
    model_name="RQ5_Ensemble_XGB",
)

# Quick check
display(lift_rq5_ensemble)


Test length: 11520
  y_test: 11520
  forensic_test_proba: 11520
  bovw_k_test_proba: 11520
  bovw_test_proba: 11520
  pca_test_proba: 11520
  cnn_test_proba: 11520
  ens_test_proba: 11520
[LIFT] Saved RQ1_Forensic_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ1_Forensic_XGB_test.csv
[LIFT] Saved RQ2_BoVW_plus_Kmeans_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ2_BoVW_plus_Kmeans_XGB_test.csv
[LIFT] Saved RQ3_BoVW_raw_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ3_BoVW_raw_XGB_test.csv
[LIFT] Saved RQ3_BoVW_PCA_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ3_BoVW_PCA_XGB_test.csv
[LIFT] Saved RQ4_CNN lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ4_CNN_test.csv
[LIFT] Saved RQ5_Ensemble_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ5_Ensemble_XGB_test.csv


,decile,n,n_fake,n_real,pct_fake,pct_real,cum_n,cum_fake,cum_fake_rate,pop_frac,...,n_fake_postproc,n_fake_splicing,n_fake_stargan_tiles,n_fake_swap_like,pct_fake_copy_move,pct_fake_inpaint,pct_fake_postproc,pct_fake_splicing,pct_fake_stargan_tiles,pct_fake_swap_like
0,1,1152,1151,1,0.999132,0.000868,1152,1151,0.199826,0.1,...,354.0,39.0,633.0,7.0,0.058210,0.044309,0.307559,0.033884,0.549957,0.006082
1,2,1152,1131,21,0.981771,0.018229,2304,2282,0.396181,0.2,...,283.0,134.0,249.0,53.0,0.204244,0.160035,0.250221,0.118479,0.220159,0.046861
2,3,1152,1064,88,0.923611,0.076389,3456,3346,0.580903,0.3,...,277.0,178.0,68.0,102.0,0.212406,0.200188,0.260338,0.167293,0.063910,0.095865
3,4,1152,956,196,0.829861,0.170139,4608,4302,0.746875,0.4,...,39.0,238.0,10.0,157.0,0.247908,0.287657,0.040795,0.248954,0.010460,0.164226
4,5,1152,740,412,0.642361,0.357639,5760,5042,0.875347,0.5,...,4.0,199.0,0.0,243.0,0.187838,0.209459,0.005405,0.268919,0.000000,0.328378
5,6,1152,482,670,0.418403,0.581597,6912,5524,0.959028,0.6,...,1.0,120.0,0.0,246.0,0.087137,0.151452,0.002075,0.248963,0.000000,0.510373
6,7,1152,205,947,0.177951,0.822049,8064,5729,0.994618,0.7,...,2.0,48.0,0.0,132.0,0.063415,0.048780,0.009756,0.234146,0.000000,0.643902
7,8,1152,28,1124,0.024306,0.975694,9216,5757,0.999479,0.8,...,0.0,4.0,0.0,18.0,0.178571,0.035714,0.000000,0.142857,0.000000,0.642857
8,9,1152,3,1149,0.002604,0.997396,10368,5760,1.000000,0.9,...,0.0,0.0,0.0,2.0,0.000000,0.333333,0.000000,0.000000,0.000000,0.666667
9,10,1152,0,1152,0.000000,1.000000,11520,5760,1.000000,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score

# ----------------------------
# Helpers
# ----------------------------
def _to_numpy(x):
    x = np.asarray(x)
    return x.reshape(-1)

def _validate_binary_y(y):
    y = _to_numpy(y).astype(int)
    uniq = np.unique(y)
    if not set(uniq).issubset({0, 1}):
        raise ValueError(f"y must be binary 0/1. Found values: {uniq}")
    return y

# ----------------------------
# McNemar (exact, binomial)
# ----------------------------
def mcnemar_exact(y_true, p_a, p_b, threshold=0.50):
    """
    Exact McNemar test using binomial test on discordant pairs.
    Compares two classifiers A and B on the SAME examples.
    """
    y_true = _validate_binary_y(y_true)
    p_a = _to_numpy(p_a)
    p_b = _to_numpy(p_b)

    pred_a = (p_a >= threshold).astype(int)
    pred_b = (p_b >= threshold).astype(int)

    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)

    # discordant counts
    n01 = int(np.sum((~correct_a) & ( correct_b)))  # A wrong, B right
    n10 = int(np.sum(( correct_a) & (~correct_b)))  # A right, B wrong
    n = n01 + n10

    if n == 0:
        return {
            "n01_Awrong_Bright": n01,
            "n10_Aright_Bwrong": n10,
            "p_value": 1.0,
            "note": "No discordant pairs; models tie on every example."
        }

    # exact two-sided p-value via binomial test with p=0.5
    try:
        from scipy.stats import binomtest
        pval = binomtest(k=min(n01, n10), n=n, p=0.5, alternative="two-sided").pvalue
    except Exception:
        # fallback: normal approx w/ continuity correction
        import math
        diff = abs(n01 - n10) - 1.0
        chi2 = (diff * diff) / n
        # p-value for chi-square(1)
        # approx using survival function of normal: p ~ 2*(1-Phi(sqrt(chi2)))
        from math import sqrt
        z = sqrt(chi2)
        # crude normal sf
        pval = 2.0 * (1.0 - 0.5 * (1.0 + math.erf(z / np.sqrt(2))))

    return {
        "n01_Awrong_Bright": n01,
        "n10_Aright_Bwrong": n10,
        "p_value": float(pval),
        "threshold": float(threshold),
        "discordant_pairs": int(n)
    }

# ----------------------------
# DeLong test for correlated AUCs (pure numpy)
# Based on the fast DeLong algorithm (Sun & Xu) commonly used in practice.
# ----------------------------
def _compute_midrank(x):
    """Computes midranks of a 1D array."""
    x = _to_numpy(x)
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        # midrank for ties
        mid = 0.5 * (i + j - 1) + 1
        T[i:j] = mid
        i = j
    out = np.empty(N, dtype=float)
    out[J] = T
    return out

def _fast_delong(predictions_sorted_transposed, label_1_count):
    """
    Fast DeLong for 2+ classifiers.
    predictions_sorted_transposed: shape (k_classifiers, n_examples), sorted by ground-truth with positives first
    label_1_count: number of positive examples
    Returns:
      aucs: shape (k_classifiers,)
      delong_cov: covariance matrix of AUC estimates, shape (k, k)
    """
    m = int(label_1_count)
    n = predictions_sorted_transposed.shape[1] - m
    k = predictions_sorted_transposed.shape[0]

    # Compute midranks for positives and negatives per classifier
    tx = np.zeros((k, m))
    ty = np.zeros((k, n))
    tz = np.zeros((k, m + n))

    for r in range(k):
        preds = predictions_sorted_transposed[r, :]
        tz[r, :] = _compute_midrank(preds)
        tx[r, :] = _compute_midrank(preds[:m])
        ty[r, :] = _compute_midrank(preds[m:])

    aucs = (tz[:, :m].sum(axis=1) / m - (m + 1) / 2.0) / n

    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m

    sx = np.cov(v01)
    sy = np.cov(v10)
    delong_cov = sx / m + sy / n

    return aucs, delong_cov

def delong_auc_pvalue(y_true, p_a, p_b):
    """
    Returns AUCs for A and B and DeLong p-value (two-sided) for difference in AUC.
    """
    y_true = _validate_binary_y(y_true)
    p_a = _to_numpy(p_a)
    p_b = _to_numpy(p_b)

    # sort with positives first
    order = np.argsort(-y_true)  # y=1 first
    y_sorted = y_true[order]
    preds = np.vstack([p_a[order], p_b[order]])

    m = int(np.sum(y_sorted))
    if m == 0 or m == len(y_sorted):
        raise ValueError("DeLong test requires both positive and negative samples in y_true.")

    aucs, cov = _fast_delong(preds, m)
    auc_a, auc_b = float(aucs[0]), float(aucs[1])

    # variance of difference
    diff = auc_b - auc_a
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var <= 0:
        # numerical edge case
        pval = 1.0
        z = 0.0
    else:
        z = diff / np.sqrt(var)
        try:
            from scipy.stats import norm
            pval = 2 * norm.sf(abs(z))
        except Exception:
            # normal sf fallback
            import math
            pval = 2.0 * (1.0 - 0.5 * (1.0 + math.erf(abs(z) / np.sqrt(2))))

    return {
        "auc_a": auc_a,
        "auc_b": auc_b,
        "auc_diff_b_minus_a": float(diff),
        "z": float(z),
        "p_value": float(pval)
    }

# ----------------------------
# Bootstrap CI for delta metrics (AUC and F1 by default)
# ----------------------------
def bootstrap_delta_metrics(y_true, p_a, p_b, n_boot=5000, seed=42, threshold=0.50):
    """
    Bootstraps the difference (B - A) in AUC and F1 with percentile 95% CI.
    Uses paired resampling of rows.
    """
    rng = np.random.default_rng(seed)
    y_true = _validate_binary_y(y_true)
    p_a = _to_numpy(p_a)
    p_b = _to_numpy(p_b)

    n = len(y_true)
    idx = np.arange(n)

    deltas_auc = []
    deltas_f1  = []

    for _ in range(n_boot):
        s = rng.choice(idx, size=n, replace=True)
        y_s = y_true[s]
        # AUC needs both classes present in the bootstrap sample
        if len(np.unique(y_s)) < 2:
            continue

        pa_s = p_a[s]
        pb_s = p_b[s]

        auc_a = roc_auc_score(y_s, pa_s)
        auc_b = roc_auc_score(y_s, pb_s)
        deltas_auc.append(auc_b - auc_a)

        f1_a = f1_score(y_s, (pa_s >= threshold).astype(int))
        f1_b = f1_score(y_s, (pb_s >= threshold).astype(int))
        deltas_f1.append(f1_b - f1_a)

    deltas_auc = np.array(deltas_auc, dtype=float)
    deltas_f1  = np.array(deltas_f1, dtype=float)

    def ci(x):
        return (float(np.quantile(x, 0.025)), float(np.quantile(x, 0.975)), float(np.mean(x)))

    auc_ci = ci(deltas_auc)
    f1_ci  = ci(deltas_f1)

    return {
        "n_boot_requested": int(n_boot),
        "n_boot_used_auc_f1": int(len(deltas_auc)),
        "threshold": float(threshold),
        "delta_auc_mean": auc_ci[2],
        "delta_auc_ci95": (auc_ci[0], auc_ci[1]),
        "delta_f1_mean": f1_ci[2],
        "delta_f1_ci95": (f1_ci[0], f1_ci[1]),
    }

# ============================
# RUN THE TESTS (RQ5 vs RQ1)
# ============================

# Your variables:
# y_test, forensic_test_proba (RQ1), ens_test_proba (RQ5)
y = y_test
p_rq1 = forensic_test_proba
p_rq5 = ens_test_proba

print("AUC RQ1 (forensic):", roc_auc_score(y, p_rq1))
print("AUC RQ5 (ensemble):", roc_auc_score(y, p_rq5))
print("F1  RQ1 (thr=0.50):", f1_score(y, (np.asarray(p_rq1) >= 0.50).astype(int)))
print("F1  RQ5 (thr=0.50):", f1_score(y, (np.asarray(p_rq5) >= 0.50).astype(int)))

# 1) DeLong test (AUC diff)
delong_res = delong_auc_pvalue(y, p_rq1, p_rq5)

# 2) McNemar exact test (paired errors) at threshold 0.50
mcnemar_res = mcnemar_exact(y, p_rq1, p_rq5, threshold=0.50)

# 3) Bootstrap CI for delta AUC and delta F1
boot_res = bootstrap_delta_metrics(y, p_rq1, p_rq5, n_boot=5000, seed=42, threshold=0.50)

print("\n=== Statistical Comparison: RQ5 vs RQ1 (holdout test set) ===")
print("DeLong AUC test:", delong_res)
print("McNemar test:", mcnemar_res)
print("Bootstrap delta metrics:", boot_res)

alpha = 0.05
print("\nDecision @ alpha=0.05:")
print("  DeLong significant? ", delong_res["p_value"] < alpha)
print("  McNemar significant?", mcnemar_res["p_value"] < alpha)
print("  Bootstrap AUC CI excludes 0? ", not (boot_res["delta_auc_ci95"][0] <= 0 <= boot_res["delta_auc_ci95"][1]))
print("  Bootstrap F1  CI excludes 0? ", not (boot_res["delta_f1_ci95"][0]  <= 0 <= boot_res["delta_f1_ci95"][1]))


AUC RQ1 (forensic): 0.9323753978587964
AUC RQ5 (ensemble): 0.9541965663580246
F1  RQ1 (thr=0.50): 0.844288927768058
F1  RQ5 (thr=0.50): 0.8785926660059464

=== Statistical Comparison: RQ5 vs RQ1 (holdout test set) ===
DeLong AUC test: {'auc_a': 0.9323753978587964, 'auc_b': 0.9541965663580245, 'auc_diff_b_minus_a': 0.021821168499228105, 'z': 14.080839278462962, 'p_value': 4.9814644968371594e-45}
McNemar test: {'n01_Awrong_Bright': 866, 'n10_Aright_Bwrong': 467, 'p_value': 4.56062411516084e-28, 'threshold': 0.5, 'discordant_pairs': 1333}
Bootstrap delta metrics: {'n_boot_requested': 5000, 'n_boot_used_auc_f1': 5000, 'threshold': 0.5, 'delta_auc_mean': 0.021854852808665153, 'delta_auc_ci95': (0.018789215048513393, 0.024974950994886315), 'delta_f1_mean': 0.03435337293605371, 'delta_f1_ci95': (0.02815859364654729, 0.04048352263555461)}

Decision @ alpha=0.05:
  DeLong significant?  True
  McNemar significant? True
  Bootstrap AUC CI excludes 0?  True
  Bootstrap F1  CI excludes 0?  True


In [ ]:
# RQ1A forensic model without file_size
forensic_cols_a = [
    "brightness", "contrast", "sharpness_l1_mean",
    "edge_density", "lap_var",
    #"file_size"
]

X_train_forensic_a = train_df[forensic_cols_a].values
X_test_forensic_a  = test_df[forensic_cols_a].values

forensic_model_a, forensic_oof_a, forensic_test_proba_a, forensic_cv_a, forensic_test_a = \
    run_xgb_cv_and_test(X_train_forensic_a, y_train,
                        X_test_forensic_a, y_test,
                        model_name="RQ1A_Forensic_XGB_without_file_size")

print("\nRQ1A forensic TEST metrics without file_size:", forensic_test_a)



=== RQ1A_Forensic_XGB_without_file_size: 5-fold CV ===
  Fold 1: AUC=0.607, F1=0.400, Acc=0.593, Prec=0.760, Rec=0.271, time=2.62s
  Fold 2: AUC=0.599, F1=0.426, Acc=0.587, Prec=0.700, Rec=0.306, time=2.69s
  Fold 3: AUC=0.602, F1=0.409, Acc=0.588, Prec=0.724, Rec=0.285, time=2.58s
  Fold 4: AUC=0.601, F1=0.431, Acc=0.587, Prec=0.694, Rec=0.312, time=5.38s
  Fold 5: AUC=0.607, F1=0.429, Acc=0.597, Prec=0.738, Rec=0.302, time=2.62s

CV summary:
  auc_mean: 0.6030
  auc_std: 0.0035
  accuracy_mean: 0.5907
  accuracy_std: 0.0039
  precision_mean: 0.7233
  precision_std: 0.0244
  recall_mean: 0.2954
  recall_std: 0.0152
  f1_mean: 0.4189
  f1_std: 0.0123
  train_time_s_mean: 3.1756
  train_time_s_std: 1.1006

=== RQ1A_Forensic_XGB_without_file_size: TEST metrics ===
  AUC=0.623, F1=0.428, Acc=0.607, Prec=0.788, Rec=0.294
  Features=5, Train_time_full=2.87s

RQ1A forensic TEST metrics without file_size: {'auc': 0.6228350453317901, 'accuracy': 0.6074652777777778, 'precision': 0.787639405204

In [ ]:
# Example: RQ1 forensic model without file_size


shap_rq1a = explain_xgb_with_shap(
    model=forensic_model_a,
    X_train=X_train_forensic_a,
    X_test=X_test_forensic_a,
    y_test=y_test,
    feature_names=forensic_cols_a,
    prefix="rq1a_forensic_xgb_without_file_size"
)


=== SHAP explainability for rq1a_forensic_xgb_without_file_size ===
  - Saved SHAP beeswarm to /content/stargan-v2/explainability/rq1a_forensic_xgb_without_file_size_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/stargan-v2/explainability/rq1a_forensic_xgb_without_file_size_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/stargan-v2/explainability/rq1a_forensic_xgb_without_file_size_shap_importance.csv
  - Saved local SHAP force plot (tp) to /content/stargan-v2/explainability/rq1a_forensic_xgb_without_file_size_local_tp_idx5760.png
  - Saved local SHAP force plot (fp) to /content/stargan-v2/explainability/rq1a_forensic_xgb_without_file_size_local_fp_idx31.png
  - Saved local SHAP force plot (fn) to /content/stargan-v2/explainability/rq1a_forensic_xgb_without_file_size_local_fn_idx6721.png


In [ ]:
# === Build lift tables on TEST for each model ===

# Sanity check that all proba arrays line up with test_df
print("Test length:", len(test_df))
print("  y_test:", len(y_test))
print("  forensic_test_proba_a:", len(forensic_test_proba_a))

lift_rq1a_forensic_without_file_size = compute_lift_table(
    y_true=y_test,
    y_score=forensic_test_proba_a,
    base_df=test_df,
    model_name="RQ1a_Forensic_XGB_without_file_size",
)

# Quick check
display(lift_rq1a_forensic_without_file_size)


Test length: 11520
  y_test: 11520
  forensic_test_proba_a: 11520
[LIFT] Saved RQ1a_Forensic_XGB_without_file_size lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ1a_Forensic_XGB_without_file_size_test.csv


,decile,n,n_fake,n_real,pct_fake,pct_real,cum_n,cum_fake,cum_fake_rate,pop_frac,...,n_fake_postproc,n_fake_splicing,n_fake_stargan_tiles,n_fake_swap_like,pct_fake_copy_move,pct_fake_inpaint,pct_fake_postproc,pct_fake_splicing,pct_fake_stargan_tiles,pct_fake_swap_like
0,1,1152,1091,61,0.947049,0.052951,1152,1091,0.189410,0.1,...,234,1,841,0,0.006416,0.007333,0.214482,0.000917,0.770852,0.000000
1,2,1152,681,471,0.591146,0.408854,2304,1772,0.307639,0.2,...,261,56,119,55,0.145374,0.133627,0.383260,0.082232,0.174743,0.080764
2,3,1152,509,643,0.441840,0.558160,3456,2281,0.396007,0.3,...,91,107,0,101,0.206287,0.206287,0.178782,0.210216,0.000000,0.198428
3,4,1152,512,640,0.444444,0.555556,4608,2793,0.484896,0.4,...,83,120,0,104,0.208984,0.191406,0.162109,0.234375,0.000000,0.203125
4,5,1152,505,647,0.438368,0.561632,5760,3298,0.572569,0.5,...,76,98,0,129,0.196040,0.203960,0.150495,0.194059,0.000000,0.255446
5,6,1152,480,672,0.416667,0.583333,6912,3778,0.655903,0.6,...,59,110,0,121,0.208333,0.187500,0.122917,0.229167,0.000000,0.252083
6,7,1152,508,644,0.440972,0.559028,8064,4286,0.744097,0.7,...,50,113,0,122,0.216535,0.222441,0.098425,0.222441,0.000000,0.240157
7,8,1152,512,640,0.444444,0.555556,9216,4798,0.832986,0.8,...,47,114,0,111,0.240234,0.228516,0.091797,0.222656,0.000000,0.216797
8,9,1152,509,643,0.441840,0.558160,10368,5307,0.921354,0.9,...,37,127,0,122,0.200393,0.237721,0.072692,0.249509,0.000000,0.239686
9,10,1152,453,699,0.393229,0.606771,11520,5760,1.000000,1.0,...,22,114,0,95,0.238411,0.251656,0.048565,0.251656,0.000000,0.209713


In [ ]:
# === Zip StarGAN-v2 'data' folder and move to Drive ===
from pathlib import Path
from datetime import datetime
import shutil, sys

SRC = Path("/content/stargan-v2/explainability")
DEST_DIR = Path("/content/drive/MyDrive/deepfake_project/output/explainability")

# Safety checks
if not SRC.exists():
    raise FileNotFoundError(f"Source folder not found: {SRC}")
if not any(SRC.rglob("*")):
    raise RuntimeError(f"Source folder is empty: {SRC}")

DEST_DIR.mkdir(parents=True, exist_ok=True)

# Timestamped archive name
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
archive_base = Path(f"/content/stargan-v2_explainability-{stamp}")  # no extension for make_archive

print(f"Creating ZIP archive from: {SRC}")
# Create ZIP (you can switch 'zip' -> 'gztar' for .tar.gz if preferred)
zip_path_str = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=SRC.parent,   # /content/stargan-v2
    base_dir=SRC.name      # data
)
archive_path = Path(zip_path_str)

# Move to Drive
final_path = DEST_DIR / archive_path.name
print(f"Moving archive to: {final_path}")
shutil.move(str(archive_path), str(final_path))

# Report
size_gb = final_path.stat().st_size / (1024**3)
print(f"✅ Archive ready: {final_path}")
print(f"   Size: {size_gb:.2f} GB")


Creating ZIP archive from: /content/stargan-v2/explainability
Moving archive to: /content/drive/MyDrive/deepfake_project/output/explainability/stargan-v2_explainability-20260216-155644.zip
✅ Archive ready: /content/drive/MyDrive/deepfake_project/output/explainability/stargan-v2_explainability-20260216-155644.zip
   Size: 0.02 GB


In [ ]:
print("Completed")

Completed
